In [9]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "statsmodels"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("\nSTDOUT:\n", result.stdout[:5000])
print("\nSTDERR:\n", result.stderr[:5000])


Return code: 0

STDOUT:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.0 MB ? eta -:--:--
   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/10.0 MB 20.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 8.9/10.0 MB 24.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 24.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/20.3 MB ? eta -:--:--
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/20.3 MB 25.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 9.2/20.3 MB 23.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 13.6/20.3 MB 22.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 18.4/20.3 MB 22.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 21.5 MB/s  0:00:00

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0

In [10]:
import importlib.util

HAS_STATSMODELS = importlib.util.find_spec("statsmodels") is not None
print("statsmodels installed:", HAS_STATSMODELS)

if HAS_STATSMODELS:
    import statsmodels.formula.api as smf
    print("statsmodels imported successfully")


statsmodels installed: True
statsmodels imported successfully


In [11]:
import pandas as pd
import numpy as np
import requests
import seaborn as sns
import matplotlib.pyplot as plt
import importlib.util

from IPython.display import display

HAS_STATSMODELS = importlib.util.find_spec("statsmodels") is not None
if HAS_STATSMODELS:
    import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

POP_PATH = "/Users/tonytony/Final Project/Data/Raw/population.csv"
LIFE_PATH = "/Users/tonytony/Final Project/Data/Raw/life_expectancy.csv"
META_PATH = "/Users/tonytony/Final Project/Data/Raw/world_bank_country_metadata.csv"

print("statsmodels installed:", HAS_STATSMODELS)
print("Population path:", POP_PATH)
print("Life Expectancy path:", LIFE_PATH)
print("Metadata path:", META_PATH)


statsmodels installed: True
Population path: /Users/tonytony/Final Project/Data/Raw/population.csv
Life Expectancy path: /Users/tonytony/Final Project/Data/Raw/life_expectancy.csv
Metadata path: /Users/tonytony/Final Project/Data/Raw/world_bank_country_metadata.csv


In [12]:
wb_meta = pd.read_csv(META_PATH)

print("Metadata shape:", wb_meta.shape)
print("Metadata columns:")
print(wb_meta.columns.tolist())

keep_cols = ["Country Code", "Country Name", "World Bank Region", "Income Level", "Lending Type"]
available_cols = [col for col in keep_cols if col in wb_meta.columns]

region_map = wb_meta[available_cols].copy()

rename_map = {
    "Country Code": "country_code",
    "Country Name": "country_name_meta",
    "World Bank Region": "wb_region",
    "Income Level": "income_level",
    "Lending Type": "lending_type"
}
region_map = region_map.rename(columns=rename_map)

region_map["country_type"] = np.where(
    region_map["wb_region"].astype(str).str.strip().eq("Aggregates"),
    "Aggregate",
    "Country/Territory"
)

print("Region mapping shape:", region_map.shape)
region_map.head()


Metadata shape: (296, 10)
Metadata columns:
['Country Code', 'ISO2 Code', 'World Bank Name', 'World Bank Region', 'Admin Region', 'Income Level', 'Lending Type', 'Capital City', 'Longitude', 'Latitude']
Region mapping shape: (296, 5)


,country_code,wb_region,income_level,lending_type,country_type
0,ABW,Latin America & Caribbean,High income,Not classified,Country/Territory
1,AFE,Aggregates,Aggregates,Aggregates,Aggregate
2,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,IDA,Country/Territory
3,AFR,Aggregates,Aggregates,Aggregates,Aggregate
4,AFW,Aggregates,Aggregates,Aggregates,Aggregate


In [18]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def build_retry_session():
    session = requests.Session()

    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"])
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session

def load_wdi(path):
    df = pd.read_csv(path, skiprows=4)
    df = df.drop(
        columns=[col for col in df.columns if str(col).startswith("Unnamed")],
        errors="ignore"
    )
    year_cols = [col for col in df.columns if str(col).isdigit()]
    df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")
    return df

def to_long(df, value_name):
    id_candidates = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
    id_vars = [col for col in id_candidates if col in df.columns]
    year_cols = [col for col in df.columns if str(col).isdigit()]

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="year",
        value_name=value_name
    )

    rename_map = {
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Indicator Name": "indicator_name",
        "Indicator Code": "indicator_code"
    }
    long_df = long_df.rename(columns=rename_map)
    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int64")
    long_df[value_name] = pd.to_numeric(long_df[value_name], errors="coerce")
    return long_df

def fetch_wb_indicator(indicator_code, value_name, start_year=1990, end_year=2024, per_page=1000):
    session = build_retry_session()
    base_url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"

    first_params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": per_page,
        "page": 1
    }

    first_response = session.get(base_url, params=first_params, timeout=60)
    first_response.raise_for_status()
    first_payload = first_response.json()

    if not isinstance(first_payload, list) or len(first_payload) < 2:
        raise ValueError(f"Unexpected API response for {indicator_code}")

    meta = first_payload[0]
    total_pages = int(meta.get("pages", 1))

    rows = []

    for page in range(1, total_pages + 1):
        params = {
            "format": "json",
            "date": f"{start_year}:{end_year}",
            "per_page": per_page,
            "page": page
        }

        response = session.get(base_url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, list) or len(payload) < 2:
            continue

        for item in payload[1]:
            country_code = item.get("countryiso3code")
            country = item.get("country", {})
            country_name = country.get("value")
            year = item.get("date")
            value = item.get("value")

            if not country_code or not year or not str(year).isdigit():
                continue

            rows.append({
                "country_code": country_code,
                "country_name_api": country_name,
                "year": int(year),
                value_name: value
            })

        print(f"{indicator_code} - page {page}/{total_pages} loaded")
        time.sleep(0.2)

    df = pd.DataFrame(rows)
    df[value_name] = pd.to_numeric(df[value_name], errors="coerce")
    return df




In [14]:
pop_df = load_wdi(POP_PATH)
pop_long = to_long(pop_df, "population_total")

pop_long = pop_long.merge(
    region_map[["country_code", "wb_region", "country_type"]],
    on="country_code",
    how="left"
)

pop_long = pop_long[
    (pop_long["country_type"] == "Country/Territory") &
    (pop_long["population_total"].notna()) &
    (pop_long["population_total"] > 0)
].copy()

pop_long = pop_long.sort_values(["country_code", "year"]).copy()
pop_long["previous_year"] = pop_long.groupby("country_code")["year"].shift(1)
pop_long["previous_population"] = pop_long.groupby("country_code")["population_total"].shift(1)

valid_gap = (pop_long["year"] - pop_long["previous_year"]) == 1

pop_long["population_growth_pct"] = np.where(
    valid_gap & (pop_long["previous_population"] > 0),
    (pop_long["population_total"] / pop_long["previous_population"] - 1) * 100,
    np.nan
)

print("Population raw shape:", pop_df.shape)
print("Population long shape:", pop_long.shape)
print("Year range:", int(pop_long["year"].min()), "-", int(pop_long["year"].max()))
print("Countries:", pop_long["country_code"].nunique())

pop_long[[
    "country_name",
    "country_code",
    "wb_region",
    "year",
    "population_total",
    "population_growth_pct"
]].head(15)


Population raw shape: (266, 69)
Population long shape: (14075, 11)
Year range: 1960 - 2024
Countries: 217


,country_name,country_code,wb_region,year,population_total,population_growth_pct
0,Aruba,ABW,Latin America & Caribbean,1960,54922.0,NaN
266,Aruba,ABW,Latin America & Caribbean,1961,55578.0,1.194421
532,Aruba,ABW,Latin America & Caribbean,1962,56320.0,1.335061
798,Aruba,ABW,Latin America & Caribbean,1963,57002.0,1.210938
1064,Aruba,ABW,Latin America & Caribbean,1964,57619.0,1.082418
1330,Aruba,ABW,Latin America & Caribbean,1965,58190.0,0.990993
1596,Aruba,ABW,Latin America & Caribbean,1966,58694.0,0.866128
1862,Aruba,ABW,Latin America & Caribbean,1967,58990.0,0.504310
2128,Aruba,ABW,Latin America & Caribbean,1968,59069.0,0.133921
2394,Aruba,ABW,Latin America & Caribbean,1969,59052.0,-0.028780


In [15]:
life_df = load_wdi(LIFE_PATH)
life_long = to_long(life_df, "life_expectancy_years")

life_long = life_long.merge(
    region_map[["country_code", "wb_region", "country_type"]],
    on="country_code",
    how="left"
)

life_long = life_long[
    (life_long["country_type"] == "Country/Territory") &
    (life_long["life_expectancy_years"].notna())
].copy()

print("Life raw shape:", life_df.shape)
print("Life long shape:", life_long.shape)
print("Latest year with life expectancy data:", int(life_long["year"].max()))

life_long[[
    "country_name",
    "country_code",
    "wb_region",
    "year",
    "life_expectancy_years"
]].head(15)


Life raw shape: (266, 69)
Life long shape: (13854, 8)
Latest year with life expectancy data: 2023


,country_name,country_code,wb_region,year,life_expectancy_years
0,Aruba,ABW,Latin America & Caribbean,1960,64.049000
2,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",1960,32.799000
4,Angola,AGO,Sub-Saharan Africa,1960,37.933000
5,Albania,ALB,Europe & Central Asia,1960,56.413000
6,Andorra,AND,Europe & Central Asia,1960,72.094000
8,United Arab Emirates,ARE,"Middle East, North Africa, Afghanistan & Pakistan",1960,50.651000
9,Argentina,ARG,Latin America & Caribbean,1960,64.242000
10,Armenia,ARM,Europe & Central Asia,1960,59.063000
11,American Samoa,ASM,East Asia & Pacific,1960,65.053000
12,Antigua and Barbuda,ATG,Latin America & Caribbean,1960,62.635000


In [19]:
gdp_growth_test = fetch_wb_indicator(
    indicator_code="NY.GDP.MKTP.KD.ZG",
    value_name="gdp_growth_pct",
    start_year=1990,
    end_year=2024,
    per_page=1000
)

print("GDP growth test shape:", gdp_growth_test.shape)
gdp_growth_test.head()


NY.GDP.MKTP.KD.ZG - page 1/10 loaded
NY.GDP.MKTP.KD.ZG - page 2/10 loaded
NY.GDP.MKTP.KD.ZG - page 3/10 loaded
NY.GDP.MKTP.KD.ZG - page 4/10 loaded
NY.GDP.MKTP.KD.ZG - page 5/10 loaded
NY.GDP.MKTP.KD.ZG - page 6/10 loaded
NY.GDP.MKTP.KD.ZG - page 7/10 loaded
NY.GDP.MKTP.KD.ZG - page 8/10 loaded
NY.GDP.MKTP.KD.ZG - page 9/10 loaded
NY.GDP.MKTP.KD.ZG - page 10/10 loaded
GDP growth test shape: (9135, 4)


,country_code,country_name_api,year,gdp_growth_pct
0,AFE,Africa Eastern and Southern,2024,2.763839
1,AFE,Africa Eastern and Southern,2023,1.931160
2,AFE,Africa Eastern and Southern,2022,3.722717
3,AFE,Africa Eastern and Southern,2021,4.578772
4,AFE,Africa Eastern and Southern,2020,-2.817572


In [20]:
INDICATORS = {
    "gdp_growth_pct": "NY.GDP.MKTP.KD.ZG",
    "gross_capital_formation_pct_gdp": "NE.GDI.TOTL.ZS",
    "trade_pct_gdp": "NE.TRD.GNFS.ZS",
}

driver_dfs = []

for value_name, indicator_code in INDICATORS.items():
    df = fetch_wb_indicator(
        indicator_code=indicator_code,
        value_name=value_name,
        start_year=1990,
        end_year=2024,
        per_page=1000
    )
    print(f"{value_name} shape:", df.shape)
    driver_dfs.append(df)


NY.GDP.MKTP.KD.ZG - page 1/10 loaded
NY.GDP.MKTP.KD.ZG - page 2/10 loaded
NY.GDP.MKTP.KD.ZG - page 3/10 loaded
NY.GDP.MKTP.KD.ZG - page 4/10 loaded
NY.GDP.MKTP.KD.ZG - page 5/10 loaded
NY.GDP.MKTP.KD.ZG - page 6/10 loaded
NY.GDP.MKTP.KD.ZG - page 7/10 loaded
NY.GDP.MKTP.KD.ZG - page 8/10 loaded
NY.GDP.MKTP.KD.ZG - page 9/10 loaded
NY.GDP.MKTP.KD.ZG - page 10/10 loaded
gdp_growth_pct shape: (9135, 4)
NE.GDI.TOTL.ZS - page 1/10 loaded
NE.GDI.TOTL.ZS - page 2/10 loaded
NE.GDI.TOTL.ZS - page 3/10 loaded
NE.GDI.TOTL.ZS - page 4/10 loaded
NE.GDI.TOTL.ZS - page 5/10 loaded
NE.GDI.TOTL.ZS - page 6/10 loaded
NE.GDI.TOTL.ZS - page 7/10 loaded
NE.GDI.TOTL.ZS - page 8/10 loaded
NE.GDI.TOTL.ZS - page 9/10 loaded
NE.GDI.TOTL.ZS - page 10/10 loaded
gross_capital_formation_pct_gdp shape: (9135, 4)
NE.TRD.GNFS.ZS - page 1/10 loaded
NE.TRD.GNFS.ZS - page 2/10 loaded
NE.TRD.GNFS.ZS - page 3/10 loaded
NE.TRD.GNFS.ZS - page 4/10 loaded
NE.TRD.GNFS.ZS - page 5/10 loaded
NE.TRD.GNFS.ZS - page 6/10 loaded
NE.